In [1]:

F = GF(11)
Rx.<x> = PolynomialRing(F)
Ry.<y> = PolynomialRing(F)
R.<x,y> = PolynomialRing(F)

# -----------------------------
# Paramètres du code 
# -----------------------------
n = 8
k = 3

xs = list(F)[:n]

f_true = Rx(x^2 + 3*x + 2)

points = [(xi, f_true(xi)) for xi in xs]

# erreurs
points = list(points)
points[2] = (points[2][0], points[2][1] + 3)
points[5] = (points[5][0], points[5][1] + 4)

# -----------------------------
# Paramètres GS
# -----------------------------
m = 2
l = 2
D = 11   # degré pondéré

# -----------------------------
# Monomes pondérés
# -----------------------------
monomials = []

for j in range(l+1):
    for i in range(D+1):
        if i + k*j <= D:
            monomials.append(x^i * y^j)

print("Nombre de monomes :", len(monomials))

# -----------------------------
# Hasse derivatives
# -----------------------------
def hasse(poly, a, b):
    return poly.derivative(x, a).derivative(y, b)

# -----------------------------
# Construction matrice
# -----------------------------
rows = []

for (xi, yi) in points:
    for a in range(m):
        for b in range(m - a):
            row = []
            for mon in monomials:
                val = hasse(mon, a, b)(x=xi, y=yi)
                row.append(val)
            rows.append(row)

M = matrix(F, rows)

print("Taille matrice :", M.nrows(), "x", M.ncols())

# -----------------------------
# Noyau (solution non triviale)
# -----------------------------
kernel = M.right_kernel()

if kernel.dimension() == 0:
    print("Pas de solution, augmenter D ou l")
    exit()

vec = kernel.basis()[0]

# -----------------------------
# Construire Q(x,y)
# -----------------------------
Q = sum(vec[i]*monomials[i] for i in range(len(monomials)))

print("\n Q(x,y) =", Q)

# -----------------------------
# Factorisation
# -----------------------------
fac = Q.factor()

print("\nFactorisation :")
print(fac)

# -----------------------------
# Extraction candidats
# -----------------------------
candidates = []

for fct, mult in fac:
    if fct.degree(y) == 1:
        a = fct.coefficient({y:1})
        b = fct.coefficient({y:0})
        f_candidate = -b/a
        candidates.append(f_candidate)

# -----------------------------
# Vérification
# -----------------------------
print("\n Candidats :")

for fc in candidates:
    count = sum(1 for (xi, yi) in points if fc(x=xi) == yi)
    print("f =", fc, "matches =", count)
#fildrage grace a la dimension ( deg(f) < k)
Vcandidates = []
for fct, mult in fac:
    if fct.degree(y) == 1:
        a = fct.coefficient({y:1})
        b = fct.coefficient({y:0})
        f_candidate = -b/a    
        if f_candidate in Rx and f_candidate.degree() < k:
            Vcandidates.append(f_candidate)
print("\n Solution trouvé :", Vcandidates)
print("\n Solution réelle :", f_true)

Nombre de monomes : 27
Taille matrice : 24 x 27

 Q(x,y) = 3*x^10 - 2*x^9 - 3*x^8*y + 5*x^8 + 4*x^7*y + 4*x^7 - 4*x^5*y^2 + 2*x^6 + 2*x^5*y + 2*x^4*y^2 + x^5 + 2*x^4*y + 4*x^3*y^2 - 3*x^4 - 4*x^3*y - x^2*y^2 + 5*x^3 + x^2*y + 5*x*y^2 + x*y + 3*y^2 - y + 1

Factorisation :
(-4) * (x - 5) * (x - 3) * (x - 2) * (-x^2 - 3*x + y - 2) * (-2*x^5 + 2*x^4 + 2*x^3 + x^2*y - 4*x^2 + 4*x*y - 5*x - 3*y - 5)

 Candidats :
f = x^2 + 3*x + 2 matches = 6
f = (2*x^5 - 2*x^4 - 2*x^3 + 4*x^2 + 5*x + 5)/(x^2 + 4*x - 3) matches = 7

 Solution trouvé : [x^2 + 3*x + 2]

 Solution réelle : x^2 + 3*x + 2
